# Documetation analysis

In [3]:
from collections import defaultdict
import pymongo 
import os
import logging


In [ ]:

def initialize_client():
    """Initialize MongoDB Client"""
    mongo_host = os.getenv('MONGO_HOST', 'localhost')
    mongo_port = os.getenv('MONGO_PORT', '27017')
    mongo_user = os.getenv('MONGO_USER')
    mongo_pass = os.getenv('MONGO_PWD')
    mongo_auth_src = os.getenv('MONGO_AUTH_SRC', 'admin')

    logging.info(f"Connecting to MongoDB at {mongo_host}:{mongo_port}")

    # Initialize MongoDB Client with AutoReconnect handling
    try:
        client = pymongo.MongoClient(
            host=[f'{mongo_host}:{mongo_port}'],
            username=mongo_user,
            password=mongo_pass,
            authSource=mongo_auth_src,
            authMechanism='SCRAM-SHA-256',
            maxPoolSize=100,
            serverSelectionTimeoutMS=5000  # Avoid indefinite hanging
        )
        return client
    except Exception as e:
        logging.error(f"MongoDB connection failed: {e}")
        raise

In [4]:
client = initialize_client()
db = client['oeb-research-software']

In [5]:
entries=db.pretoolsDev.find()
entries = list(entries)

In [6]:
print(entries[0])

{'_id': 'github/validatefastq/None/v0.1.1', 'created_at': '2025-03-04T22:22:48.094915', 'created_by': 'https://gitlab.bsc.es/None/None/-/commit/None', 'created_logs': 'local', 'last_updated_at': '2025-03-04T22:22:48.094915', 'updated_by': 'https://gitlab.bsc.es/None/None/-/commit/None', 'updated_logs': 'local', 'source': [{'collection': 'alambiqueDev', 'id': 'github/validatefastq', 'source_url': 'https://github.com/biopet/validatefastq'}], 'data': {'name': 'validatefastq', 'type': None, 'version': ['v0.1.1'], 'label': ['validatefastq'], 'links': [], 'webpage': [], 'download': [], 'repository': [{'url': 'https://github.com/biopet/validatefastq', 'kind': 'github', 'source_hasAnonymousAccess': None, 'source_isDownloadRegistered': None, 'source_isFree': None, 'source_isRepoAccessible': None}], 'operating_system': [], 'source_code': [], 'https': False, 'ssl': False, 'operational': False, 'bioschemas': False, 'source': ['github'], 'edam_topics': [], 'edam_operations': [], 'description': [], 

## Integration

### Set A and set B

In [8]:
main_sources = ['biotools','galaxy','galaxy_metadata', 'toolshed', 'bioconda', 'bioconda_recipes' ]
repository_sources = ['github', 'sourceforge', 'bioconductor']
data_entries_set_a = [data_entry for data_entry in entries if data_entry['data']['source'][0] in main_sources]
data_entries_set_b = [data_entry for data_entry in entries if data_entry['data']['source'][0] in repository_sources]

In [9]:
print(len(data_entries_set_a))
print(len(data_entries_set_b))

62567
10424


In [22]:
def group_by_key(instances):
    grouped_instances = {}
    for inst in instances:
        key = f"{inst['data']['name'].lower()}/{inst['data']['type']}"  # Grouping key
        if key not in grouped_instances:
            grouped_instances[key] = { "instances" : [inst] }
        else:
            grouped_instances[key]['instances'].append(inst)
    return grouped_instances

In [23]:
grouped_by_key_a = group_by_key(data_entries_set_a)
print(list(grouped_by_key_a.keys())[:10])

['aurocch/lib', 'ps2-v3/web', 'ps2_-_v3/web', 'ps2_-_v38938/web', 'ps2/web', 'net_bio/lib', '1000genomes/db', '1000genomes/web', '1000genomes_id_history_converter/web', '1000genomes_vcf2ped/web']


In [26]:
for key in grouped_by_key_a.keys():
    links = []
    for inst in grouped_by_key_a[key]['instances']:
        for repo in inst['data']['repository']:
            if repo.get('url'):
                links.append(repo['url'])
        
        # add webpage
        if inst['data'].get('webpage'):
            for link in inst['data']['webpage']:
                links.append(link)
    
    grouped_by_key_a[key]['links'] = links


In [28]:
filtered_data_entries_set_b = []
for instance in data_entries_set_b:
    source_url = instance['source'][0]['source_url']
    for key in grouped_by_key_a:
        if source_url in grouped_by_key_a[key]['links']:
            grouped_by_key_a[key]['instances'].append(instance)
        
        else:
            filtered_data_entries_set_b.append(instance)

In [29]:
pseudogrouped_grouped_b = {}
for inst in filtered_data_entries_set_b:
    key = f"{inst['data']['name'].lower()}/{inst['data']['type']}"  # Grouping key
    if key not in pseudogrouped_grouped_b:
        pseudogrouped_grouped_b[key] = { "instances" : [inst] }
    else:
        pseudogrouped_grouped_b[key]['instances'].append(inst)

for key in pseudogrouped_grouped_b.keys():
    links = []
    for inst in pseudogrouped_grouped_b[key]['instances']:
        for repo in inst['data']['repository']:
            if repo.get('url'):
                links.append(repo['url'])
        
        # add webpage
        if inst['data'].get('webpage'):
            for link in inst['data']['webpage']:
                links.append(link)
    
    pseudogrouped_grouped_b[key]['links'] = links

In [ ]:
# Initialize a dictionary to map links to software keys
link_to_keys = defaultdict(set)

# Populate the dictionary
for key, data in pseudogrouped_grouped_b.items():
    for link in data['links']:
        link_to_keys[link].add(key)

# Initialize groups
grouped_by_key_b = {}

# Merge software entries that share links
for key, data in pseudogrouped_grouped_b.items():
    # Find all keys that share at least one link
    related_keys = set()
    for link in data['links']:
        related_keys.update(link_to_keys[link])
    
    # Aggregate instances for all related keys
    all_instances = []
    for related_key in related_keys:
        all_instances.extend(pseudogrouped_grouped_b[related_key]['instances'])
    
    # Store the merged result
    for related_key in related_keys:
        grouped_by_key_b[related_key] = {"instances": all_instances}


NameError: name 'pseudogrouped_grouped_b' is not defined